In [1]:
import torch

print("="*50)
print("🖥️  AUTONOMOUS MULTI-AGENT PLATFORM")
print("     Refactor Agent — Resume Training")
print("="*50)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name   = torch.cuda.get_device_name(i)
        memory = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"✅ GPU {i}: {name} ({memory:.1f} GB)")
else:
    print("❌ No GPU found!")

print(f"✅ PyTorch: {torch.__version__}")

🖥️  AUTONOMOUS MULTI-AGENT PLATFORM
     Refactor Agent — Resume Training
✅ GPU 0: Tesla T4 (15.6 GB)
✅ GPU 1: Tesla T4 (15.6 GB)
✅ PyTorch: 2.9.0+cu126


In [2]:
!pip install transformers==4.41.0 -q
!pip install peft==0.10.0 -q
!pip install accelerate==0.29.0 -q

print("✅ Libraries installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 5.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 6.5 MB/s eta 0:00:0000:01
✅ Libraries installed!


In [7]:
import os, json

# ── Correct exact paths ───────────────────────────────
DATASET_DIR    = "/kaggle/input/datasets/adityapandit07/multiagent-refactor-doc-dataset"
CHECKPOINT_DIR = "/kaggle/input/datasets/adityapandit07/multiagent-refactor-checkpoint"
OUTPUT_DIR     = "/kaggle/working/models/refactor_agent"
SAVE_PATH      = "/kaggle/working/refactor_agent_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_PATH,  exist_ok=True)

print("✅ Paths configured!")
print(f"   Dataset:    {DATASET_DIR}")
print(f"   Checkpoint: {CHECKPOINT_DIR}")
print(f"   Output:     {OUTPUT_DIR}")
print(f"   Save:       {SAVE_PATH}")

✅ Paths configured!
   Dataset:    /kaggle/input/datasets/adityapandit07/multiagent-refactor-doc-dataset
   Checkpoint: /kaggle/input/datasets/adityapandit07/multiagent-refactor-checkpoint
   Output:     /kaggle/working/models/refactor_agent
   Save:       /kaggle/working/refactor_agent_final


In [8]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json(f"{DATASET_DIR}/rcct_train.json")
val_data   = load_json(f"{DATASET_DIR}/rcct_val.json")

print(f"✅ Train: {len(train_data)} examples")
print(f"✅ Val:   {len(val_data)} examples")

✅ Train: 13756 examples
✅ Val:   1719 examples


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "Salesforce/codet5p-770m"

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizer loaded!")

print("⏳ Loading base model...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"✅ Base model loaded!")

⏳ Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded!
⏳ Loading base model...


config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

✅ Base model loaded!


In [11]:
from peft import PeftModel

print("⏳ Loading Epoch 2 LoRA checkpoint...")

model = PeftModel.from_pretrained(
    model,
    CHECKPOINT_DIR,
    is_trainable=True        # ✅ This is the key fix — unfreezes LoRA weights
)

model.enable_input_require_grads()

# Cast LoRA weights to FP32
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()

⏳ Loading Epoch 2 LoRA checkpoint...
trainable params: 4,718,592 || all params: 742,358,016 || trainable%: 0.6356221524251716


In [12]:
from torch.utils.data import Dataset

MAX_LEN = 256

class CodeDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data      = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        inputs = self.tokenizer(
            item["input"],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        targets = self.tokenizer(
            item["output"],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels":         labels
        }

train_dataset = CodeDataset(train_data, tokenizer)
val_dataset   = CodeDataset(val_data,   tokenizer)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}")

✅ Train: 13756 | Val: 1719


In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ✅ Only run Epoch 3 — we already did Epochs 1 & 2
    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=3e-4,
    warmup_steps=50,       # Less warmup since resuming
    weight_decay=0.01,

    fp16=False,

    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,

    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("✅ Trainer ready — will run Epoch 3 only!")

2026-03-07 19:08:29.824433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772910510.016049      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772910510.070858      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772910510.521381      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772910510.521423      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772910510.521425      55 computation_placer.cc:177] computation placer alr

✅ Trainer ready — will run Epoch 3 only!


In [14]:
print("🚀 Autonomous Multi-Agent Platform — Refactor Agent")
print("📊 Running Epoch 3 (final epoch)...\n")

trainer.train()

print("\n✅ Epoch 3 complete! Training finished!")

🚀 Autonomous Multi-Agent Platform — Refactor Agent
📊 Running Epoch 3 (final epoch)...



/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:646: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
0,0.113000,0.109985


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



✅ Epoch 3 complete! Training finished!


In [15]:
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Refactor Agent saved!")
print(f"\n📊 Training Summary:")
print(f"   Epoch 1: Train 0.1196 | Val 0.1122  (Colab)")
print(f"   Epoch 2: Train 0.1104 | Val 0.1074  (Colab)")
print(f"   Epoch 3: (check above output)         (Kaggle)")
print(f"\n📌 Download from: Output tab → refactor_agent_final/")

✅ Refactor Agent saved!

📊 Training Summary:
   Epoch 1: Train 0.1196 | Val 0.1122  (Colab)
   Epoch 2: Train 0.1104 | Val 0.1074  (Colab)
   Epoch 3: (check above output)         (Kaggle)

📌 Download from: Output tab → refactor_agent_final/


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [16]:
# Save output as a Kaggle dataset so you can download as ZIP
import shutil

# Zip the entire model folder
shutil.make_archive(
    "/kaggle/working/refactor_agent_final",  # output zip name
    "zip",                                    # format
    "/kaggle/working",                        # root dir
    "refactor_agent_final"                    # folder to zip
)

print("✅ Zipped! File: /kaggle/working/refactor_agent_final.zip")
print("📌 Go to Output tab → refactor_agent_final.zip → click ⬇️ to download")

✅ Zipped! File: /kaggle/working/refactor_agent_final.zip
📌 Go to Output tab → refactor_agent_final.zip → click ⬇️ to download
